# **Raw dataset construction**

We gather panel data (country/year) on wellbeing, inequality, material consumption and emissions from various official sources listed below:

## Data Sources

| Dataset | Coverage | Source | Purpose in Project | Original Format
|---|---|---|---|---|
| OWID CO₂ Data | Annual country-level data (varies by variable; broad coverage 1900–2023) | https://github.com/owid/co2-data | Base dataset | Long |
| OWID Energy Data | Annual country-level energy indicators | https://github.com/owid/energy-data | Supplementary energy transition indicators | Long |
| Material Footprint per Capita | 2012–2021 | https://www.kaggle.com/datasets/iamsouravbanerjee/material-footprint-per-capita-by-country | Per-capita material consumption indicator | Wide |
| World Bank GINI Index | 1992–2025 (sparse by country) | https://data.worldbank.org/indicator/SI.POV.GINI | Income inequality indicator | Wide |
| World Happiness Index | 2013–2023 | https://www.kaggle.com/datasets/simonaasm/world-happiness-index-by-reports-2013-2023 | Subjective wellbeing indicator | Long |

## Temporal Overlap

The main datasets overlap consistently between **2013 and 2021**, giving a usable multi-year comparison window for merged analysis.

---

## Core Theoretical Variables from OWID CO₂ and Energy Datasets

| Variable | Description | Analytical Purpose |
|---|---|---|
| `population` | Total population of the country | Useful for weighting and validating per-capita indicators |
| `gdp` | Gross Domestic Product | Main macroeconomic control variable |
| `consumption_co2_per_capita` | Consumption-based CO₂ emissions per capita | Key variable linking emissions to lifestyles and consumption patterns |
| `co2_per_capita` | Production-based CO₂ emissions per capita | Used for comparison with consumption emissions and emissions-gap calculations |
| `renewables_share_energy` | Share of primary energy consumption from renewable sources | Indicator of energy transition and decarbonization |
| `energy_per_capita` | Primary energy consumption per capita | Proxy for energy intensity and resource use |

---

## Identifier and Merge Variables

| Variable | Description | Role in Merge |
|---|---|---|
| `iso_code` | ISO 3-letter country code | Main country-level merge key |
| `country` | Full country name | Human-readable country identifier |
| `year` | Observation year | Temporal merge key for panel structure |

---

## Conceptual Focus of the Project

This project investigates whether countries can achieve relatively high levels of wellbeing while maintaining lower levels of environmental and material throughput.

The analysis combines:
- wellbeing indicators,
- inequality measures,
- emissions data,
- renewable energy transition metrics,
- and material footprint indicators

to explore possible forms of:
- sustainable wellbeing,
- ecological efficiency,
- and partial decoupling between quality of life and material consumption.

### Strategy summary:
1. Import the raw datasets independently.
2. Convert wide data to long format.
3. Check temporal coverage per dataset.
4. Keep only overlapping window accross all datasets
5. Check and normalise merge keys accross datasets (iso, country, year).
6. From OWID CO2 and Energy data, pick selected columns to keep.
7. Merge by country (iso) and year keys to create the full raw dataset to be cleaned.

# 1. Import the necessary libraries

In [1]:
import pandas as pd

# 2. Load the data

In [13]:
# Long format data
owid_co2_df = pd.read_csv('../data/raw/_owid_co2_data.csv')
owid_energy_df = pd.read_csv('../data/raw/owid_energy_data.csv')
hi_df = pd.read_csv('../data/raw/world_happiness_index_data.csv')

# Wide format data has to be converted to long prior to any merging
mf_df_wide = pd.read_csv('../data/raw/material_footprint_data_wide.csv')
gini_df_wide = pd.read_csv('../data/raw/gini_data_wide.csv')

# We confirm long/wide data formats have been correctly identified
print("OWID CO2 DataFrame shape:", owid_co2_df.shape)
display(owid_co2_df.head())

print("OWID Energy DataFrame shape:", owid_energy_df.shape)
display(owid_energy_df.head())

print("World Happiness Index DataFrame shape:", hi_df.shape)
display(hi_df.head())

print("Material Footprint DataFrame shape:", mf_df_wide.shape)
display(mf_df_wide.head())

print("Gini DataFrame shape:", gini_df_wide.shape)
display(gini_df_wide.head())


OWID CO2 DataFrame shape: (50411, 79)


,country,year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,...,share_global_other_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share
0,Afghanistan,1750,AFG,2802560.0,NaN,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,1751,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,1752,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,1753,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,1754,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


OWID Energy DataFrame shape: (23377, 130)


,country,year,iso_code,population,gdp,biofuel_cons_change_pct,biofuel_cons_change_twh,biofuel_cons_per_capita,biofuel_consumption,biofuel_elec_per_capita,...,solar_share_elec,solar_share_energy,wind_cons_change_pct,wind_cons_change_twh,wind_consumption,wind_elec_per_capita,wind_electricity,wind_energy_per_capita,wind_share_elec,wind_share_energy
0,ASEAN (Ember),2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
1,ASEAN (Ember),2001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
2,ASEAN (Ember),2002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
3,ASEAN (Ember),2003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
4,ASEAN (Ember),2004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN


World Happiness Index DataFrame shape: (1670, 4)


,Country,Year,Index,Rank
0,Afghanistan,2013,4.040,143.0
1,Afghanistan,2015,3.575,153.0
2,Afghanistan,2016,3.360,154.0
3,Afghanistan,2017,3.794,141.0
4,Afghanistan,2018,3.632,145.0


Material Footprint DataFrame shape: (195, 39)


,ISO3,Country,Continent,Hemisphere,Human Development Groups,UNDP Developing Regions,HDI Rank (2021),Material footprint per capita (tonnes) (1990),Material footprint per capita (tonnes) (1991),Material footprint per capita (tonnes) (1992),...,Material footprint per capita (tonnes) (2012),Material footprint per capita (tonnes) (2013),Material footprint per capita (tonnes) (2014),Material footprint per capita (tonnes) (2015),Material footprint per capita (tonnes) (2016),Material footprint per capita (tonnes) (2017),Material footprint per capita (tonnes) (2018),Material footprint per capita (tonnes) (2019),Material footprint per capita (tonnes) (2020),Material footprint per capita (tonnes) (2021)
0,AFG,Afghanistan,Asia,Northern Hemisphere,Low,SA,180.0,2.33,2.28,2.35,...,1.86,1.88,1.66,1.62,1.66,1.41,1.32,1.38,1.38,1.38
1,AGO,Angola,Africa,Southern Hemisphere,Medium,SSA,148.0,2.44,2.66,4.67,...,4.09,4.53,3.97,3.59,2.79,2.64,2.28,2.18,2.18,2.18
2,ALB,Albania,Europe,Northern Hemisphere,High,ECA,67.0,6.63,5.91,5.65,...,12.44,11.49,13.14,12.61,14.39,14.46,12.85,12.96,12.96,12.96
3,AND,Andorra,Europe,Northern Hemisphere,Very High,NaN,40.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ARE,United Arab Emirates,Asia,Northern Hemisphere,Very High,AS,26.0,64.75,34.47,33.09,...,49.56,49.68,55.49,59.76,64.95,75.61,65.97,68.95,68.95,68.95


Gini DataFrame shape: (266, 71)


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,Gini index,SI.POV.GINI,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,Gini index,SI.POV.GINI,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,Gini index,SI.POV.GINI,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Africa Western and Central,AFW,Gini index,SI.POV.GINI,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Angola,AGO,Gini index,SI.POV.GINI,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,51.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
